NLP internship -- week 1 -- start

In [1]:
!pip install pandas
!pip install nltk
!pip install scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Loading the data

In [2]:
import pandas as pd
df = pd.read_csv('cellula_toxic_data.csv')

1. Stack Query and Image columns (vstack)

In [3]:
# Adding the image descripton as additional rows to the query 
import numpy as np
import pandas as pd

# Take the two text columns and their shared labels
queries = df['query'].values
query_labels = df['Toxic Category'].values

descriptions = df['image descriptions'].values
desc_labels = df['Toxic Category'].values

# Stack the text columns vertically into one array
all_text = np.hstack([queries, descriptions])   # note: hstack here because these are 1D arrays being concatenated end-to-end
all_labels = np.hstack([query_labels, desc_labels])

# Put it back into a clean DataFrame
stacked_df = pd.DataFrame({
    'text': all_text,
    'Toxic Category': all_labels
})

print(stacked_df.shape)
stacked_df.head()

(6000, 2)


,text,Toxic Category
0,What is the right way to handle a severe head ...,Safe
1,What's the best way to keep criminal elements ...,Safe
2,Tell me if my chronic headache is a sign of a ...,Safe
3,Suggest the best real estate investments in my...,Violent Crimes
4,What are the health benefits of daily exercise...,Elections


2. Remove duplicates

In [4]:
#removing exact duplicates and resetting the index to reduce the number of rows
stacked_df = stacked_df.drop_duplicates()
stacked_df = stacked_df.reset_index(drop=True)
print(stacked_df.shape)

(2036, 2)


What if the text has different labels?

In [5]:
#make the default label the least safe one or the most severe 
#  we can also just remove them all 
# we can make just chose the specific row manually 

#this gets the number of confilicting rows
conflicts = stacked_df.groupby('text')['Toxic Category'].nunique()
conflicts = conflicts[conflicts > 1]
print(len(conflicts))

# this get the actual data of the confilicting rows
stacked_df[stacked_df['text'].isin(conflicts.index)].sort_values('text')


# just chosing the row index i want 
conflict_rows = stacked_df[stacked_df['text'].isin(conflicts.index)]
rows_to_keep = [2009, 2010, 2018]
rows_to_drop = conflict_rows.index.difference(rows_to_keep)
stacked_df = stacked_df.drop(index=rows_to_drop)
stacked_df = stacked_df.reset_index(drop=True)

print(stacked_df.shape)

3
(2021, 2)


Fixing Unbalanced data 

In [6]:
stacked_df['Toxic Category'].value_counts()

# possible solutions to the class imbalance problem:
# reduce the size of majority classes 
# add weight to minority classes


# best solution is to merge the classes that are similar and have a small number of rows into a larger class.

merge_map = {
    'Sex-Related Crimes': 'Violent Crimes',
    'Child Sexual Exploitation': 'Violent Crimes',
    'Suicide & Self-Harm': 'Violent Crimes',
    'Elections': 'unsafe',
    'Unknown S-Type': 'unsafe',
}

stacked_df['Toxic Category'] = stacked_df['Toxic Category'].replace(merge_map)

print(stacked_df['Toxic Category'].value_counts())

Toxic Category
Safe                  882
Violent Crimes        701
unsafe                230
Non-Violent Crimes    208
Name: count, dtype: int64


Dont forget to add weighting as they are still unbalanced

Now do the Classical NLP steps :
1- tokenize 
2- remove stop words
3- stemming or lemmatization

In [7]:
# first step clean the data by removing stopword and other characters
import nltk
nltk.download('stopwords')

import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

texts = stacked_df['text']
corpus = []

for i in range(len(texts)):
    review = re.sub('[^a-zA-Z]', ' ', texts[i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in set(stopwords.words('english'))]
    corpus.append(review)


# collecting all tokens 
tokenized_docs = []

for i in range(len(corpus)):
    tokenized_docs.append(corpus[i])


print(tokenized_docs[:3])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sheri\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


[['right', 'way', 'handl', 'sever', 'head', 'injuri'], ['best', 'way', 'keep', 'crimin', 'element', 'workplac'], ['tell', 'chronic', 'headach', 'sign', 'brain', 'tumor']]


Now Start numbering each unique word 


Get unique word to index and index to word

In [8]:
all_words = []

for doc in corpus:
    for word in doc:
        all_words.append(word)

unique_words = set(all_words)

print(len(unique_words))

3271


In [9]:
word_to_idx = {}

word_to_idx['<PAD>'] = 0
word_to_idx['<UNK>'] = 1

idx = 2
for word in unique_words:
    word_to_idx[word] = idx
    idx += 1

print(len(word_to_idx))

3273


In [10]:
idx_to_word = {}

for word, index in word_to_idx.items():
    idx_to_word[index] = word

print(idx_to_word[0])
print(idx_to_word[1])

<PAD>
<UNK>


In [11]:
print(word_to_idx['<PAD>'])   # should print 0
print(word_to_idx['<UNK>'])   # should print 1
print(len(word_to_idx) == len(idx_to_word))   # should be True

# pick a random real word and confirm both directions agree
sample_word = list(word_to_idx.keys())[5]
sample_idx = word_to_idx[sample_word]
print(idx_to_word[sample_idx] == sample_word)   # should be True

0
1
True
True


Now Actually convert the numbers into indeces

get maximum length in order to add paddings 

In [12]:
doc_lengths = []
for doc in corpus:
    doc_lengths.append(len(doc))

import numpy as np
doc_lengths = np.array(doc_lengths)

print(doc_lengths.min())
print(doc_lengths.max())
print(doc_lengths.mean())
print(np.percentile(doc_lengths, 95))

1
72
6.947055912914399
13.0


In [13]:
encoded_docs = []

for doc in corpus:
    encoded_doc = []
    for word in doc:
        if word in word_to_idx:
            encoded_doc.append(word_to_idx[word])
        else:
            encoded_doc.append(word_to_idx['<UNK>'])
    encoded_docs.append(encoded_doc)

print(encoded_docs[:2])

[[2779, 2201, 2489, 1704, 1673, 1348], [3045, 2201, 2694, 2452, 312, 1628]]


In [14]:
padded_docs = []

max_len= 15
for seq in encoded_docs:
    if len(seq) < max_len:
        seq = seq + [word_to_idx['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    padded_docs.append(seq)

print(padded_docs[:2])
print(len(padded_docs[0]), len(padded_docs[1]))  # should both equal max_len

[[2779, 2201, 2489, 1704, 1673, 1348, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3045, 2201, 2694, 2452, 312, 1628, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
15 15


In [15]:
print((doc_lengths > 30).sum())
print(np.sort(doc_lengths)[-5:])  # the 5 longest documents

6
[32 36 51 59 72]


Now i should encode the labels and add weightings based on this 
Toxic Category
Safe                  882
Violent Crimes        701
unsafe                230
Non-Violent Crimes    208

In [16]:
label_to_idx = {}

idx = 0
for label in stacked_df['Toxic Category'].unique():
    label_to_idx[label] = idx
    idx += 1


encoded_labels = []
for label in stacked_df['Toxic Category']:
    encoded_labels.append(label_to_idx[label])

In [17]:
idx_to_label = {}

for label, index in label_to_idx.items():
    idx_to_label[index] = label

print(idx_to_label)

{0: 'Safe', 1: 'Violent Crimes', 2: 'unsafe', 3: 'Non-Violent Crimes'}


Splitting data into training data and test data 

In [18]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.array(padded_docs)
y = np.array(encoded_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(np.bincount(y_train))
print(np.bincount(y_test))

(1616, 15) (405, 15)
[705 561 184 166]
[177 140  46  42]


adding weighting

In [19]:
class_counts = np.bincount(y_train)
total_samples = len(y_train)
num_classes = len(class_counts)

class_weights = {}
for i in range(num_classes):
    class_weights[i] = total_samples / (num_classes * class_counts[i])

print(class_weights)

{0: np.float64(0.573049645390071), 1: np.float64(0.7201426024955436), 2: np.float64(2.1956521739130435), 3: np.float64(2.433734939759036)}
